<center><img src="https://i.pinimg.com/webp/1200x/2e/2f/af/2e2fafb318cf4a21da25dada8bc6a73e.webp" width="1200" height="400"></center>

## <b>1 <span style='color:#F1A424'>|</span> LSTM 마스터 가자..
</b> 

<div style="color:white;display:fill;border-radius:8px;font-size:100%; letter-spacing:1.0px;"><p style="padding: 5px;color:white;text-align:left;"><b><span style='color:#F1A424'>WHAT WE WILL DO IN THIS SECTION</span></b></p></div>

- 시작일 : 2026-06-11 목요일 


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.1 | </span></span></b> 문제 : RNN의 Gradient Vanishing</b></p></div>

1. 문제의 원인 : 
**1 step에서 weight는 여러 번(time step 수만큼)에 걸쳐 업데이트 된다**
<br>

    즉 백번째 시점에 도달하면 1번째 단어는 흔적도 없이 희석됨.<br> 장기 의존성(Long-Term Dependency)을 전혀 기억하지 못하는 기억력 소실 문제

    $$\text{최종 Gradient} = \nabla W^{(t=3)} + \nabla W^{(t=2)} + \nabla W^{(t=1)}$$

    초기 시점 가중치의 gradient = 0 에 가까워짐. 거의 소멸. 즉 초기 시점 가중치가 업데이트가 전혀 안됨.
---
2. Time step이 길어지면 레이어가 쌓이는 것과 같다.<br>
Time step이 길어지면 초기 Sequence에 정보가 hidden state에서 점점 사라지는 **기억력 소실문제**가 발생한다.

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.2 | 
</span></span></b> LSTM 모델 (Long Short-Term Memory)  </b></p></div>

- 핵심 : Cell state : gradient vanishing 문제를 해결해줌 
- cell state 를 만들어서 내가 필요한 것만 기억하자 ➔ 중요한 거는 메모리에 두자
- 바로 전 time step, 전체 처리 결과를 동시에 받는다.
- 현재 입력된 데이터가 얼마나 중요한가

- gate 가 무슨 기준으로 ? ➔ 또 다른 데이터를 바탕으로 결정함 ➔ 결국 시그모이드. ➔ 0-1사이로 정해짐 이 값이 게이트 오픈의 비율이됨.

- Gate : 
    -  **Forget gate**
    -  **Input gate**
    -  **Output gate**
<img src="https://media.licdn.com/dms/image/v2/D4D12AQF7aHYXD7SINQ/article-cover_image-shrink_600_2000/article-cover_image-shrink_600_2000/0/1721761416816?e=2147483647&v=beta&t=B5_d1-po8kcVseMpzGjJ_69vYypHwZunvAUjGQ3v3YU">
    

**1단계: 망각 게이트 (Forget Gate) — $f_t$ 영역** <br>

- $x_t$ : 현재 시점($t$)의 새로운 입력 데이터 
- $h_{t-1}$ : 바로 전 시점의 은닉 상태(Hidden State)<br> 단기 기억이자 직전 시점의 출력값 
- 두 값을 시그모이드(sig)함수로 넣음 ➔ 현재와 직전 시점(output : 0-1)
    - ➔ 이 결과값이 $f_t$ : 과거 기억에서 지울 만큼의 비율
- 노란색 ⊗ 에서 $C_{t-1}$에 $f_t$이 곱해짐.
    - $f_t$가 0에 가까우면 $\rightarrow$ "과거 기억($C_{t-1}$)은 이제 쓸모없으니 싹 지워라!"
    - $f_t$가 1에 가까우면 $\rightarrow$ "과거 기억 원본 그대로 온전히 보존해라!"

**2단계: 입력 게이트 (Input Gate) — $i_t$ 및 $\tilde{C}_t$ 영역** <br>
- 입력데이터 중에서 진짜 가치있고 중요한 정보만 필터링
- [sig] : $x_t$와 $h_{t-1}$를 두번째 시그모이드에 넣음 ➔ $i_t$(output:0-1)<br>➔ **새로들어온 정보 중에 어떤 것을 기억할지** 
- [tanh] : $x_t$와 $h_{t-1}$를 하이퍼볼릭 탄젠트에 넣음. ➔  $\tilde{C}_t$ <br> 이번 시점에서 C에 추가하고 싶은 새로운 기억의 후보군 

**3단계: 세포 상태 업데이트 (Cell State Update)** <br>
- 과거 기억의 축소본 : $f_t$ * C_{t-1} ➔ 현재시점의 최종 장기기억 $C_t$
- 검증된 현재 새 정보 : $i_t$ * $\tilde{C}_t$
- 즉 장기기억($C_t$) + 검증된 새로운 핵심 정보를 더하기 연산
- 핵심 :
    - 과거 기억에 곱셈 x, 현재 정보를 더해주는 구조(+), 역전파가 일어날 때 오차가 거꾸로 거슬러 올라가면 기울기 소실x 
    - 이렇게 완벽하게 업데이트된 $C_t$는 다음 시점($t+1$)의 과거 기억으로 

**4단계: 출력 게이트 (Output Gate) — $o_t$ 및 $h_t$ 영역** <br>    
- $x_t$와 $h_{t-1}$를 마지막 sig에 통과 ➔ ot 를 만듬.
- 3단계에서 완성된 현재시점의 최종 장기기억 $C_t$ 를 tanh 함수에 통과 ➔ 값의 범위를 (-1~1 사이로 정리) 
- 장기기억과 출력(ot) 를 곱해줌 ➔ 최종 출력물이자 단기기억인 ht 완성
- 다음 시점의 입력으로 다시씀(ht-1) or many to one 구조라면 최종 예측층 (linear layer)로 들어가게됨


**게이트 가중치 정리!!**
1. $W_f$ (망각 게이트 가중치) : 
    - 과거 기억을 얼마나 버릴지 결정하는 비율 계산기
    - sigmoid ➔ [0 - 1]
2. $W_i$ (입력 게이트 가중치) : 
    - 새롭게 들어온 단어의 영향력 계산기
    - sigmoid ➔ [0 - 1]
3. $W_c$ (새 기억 후보 가중치):
    - 새로운 단어("소름")의 문맥적 의미 벡터 산출 
    - tanh ➔ [-1 ~ + 1]

4. $W_o$ (출력 게이트 가중치):
    - 과거 기억에 저장된 전체 정보 중 당장 지금 시점에 얼마나 노출할지
    - tanh 


1. Forget gate ➔ 얼만큼 잊어버릴 지 결정하자(맨왼측 노란색에 반영할 
    - input : 
        - $x_t$ : 현재 시점($t$)의 새로운 입력 데이터 
        - $h_{t-1}$ : 바로 전 시점의 은닉 상태(Hidden State)<br> 단기 기억이자 직전 시점의 출력값
        - 시그모이드 ➔ 0.~ 두 입력에 대한 결과로 산출된 값.  ( 현재와 직전 시점 ) 을 평생 아 
- 1.0에 가깝다 ➔ 과거 C의 기억에 1을 곱함 ➔ 엄청 중요
- 0 에 가깝다 ➔ 과거 C의 기억을 0 ➔ 과거 특징값을 지워버림 ➔ 현재*직전이 더 가중치가 크다.
- 0.5 ➔ 반정도 지우고 반반 가져가.
    - Output : 
        - $C_{t-1}$: 과거의 세포상태 ➔ long term 

2. Input Gate : 
    intput : Wix*Xt 


시그모이드 ➔ 0.~ 두 입력에 대한 결과로 산출된 값.  ( 현재와 직전 시점 ) 
을 평생 아 
- 1.0에 가깝다 ➔ 과거 C의 기억에 1을 곱함 ➔ 엄청 중요
- 0 에 가깝다 ➔ 과거 C의 기억을 0 ➔ 과거 특징값을 지워버림 ➔ 현재*직전이 더 가중치가 크다.
- 0.5 ➔ 반정도 지우고 반반 가져가.

forget gate : Wx * wt + Wh*Wht-1 + b



<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.3 | </span></span></b>Pytorch LSTM 실습</b></p></div>

**nn.LSTM의 파라미터** <br>
- input_size: 입력 데이터의 shape
- hidden_size: Layer의 Hidden size
- num_layers: 몇층으로 Layer을 쌓을지 개수
- batch_first:
    - True - (batch, seqence len, ..)
    - False - (sequencelen, batch, ..)
    - Default: False
- dropout: Dropout rate 비율 (layer 2개 이상 쌓았을 때)
- bidirectional:  양방향 적용 여부. Default: False


**추론시 Input tensor 구조** <br>
- Input_data, (Hidden_state, Cell_state)
- Input_data의 shape
    - **(sequence_length, batch_size, input_feature_shape)**
- (Hidden_state, Cell_state)는 생략시 0 입력된다.
    - Hidden_state의 shape
        - **(D * layer수, batch_size, hidden_size)**
        - D: 양방향(bidirectional) 이면 2 아니면 1
    - Cell_state의 shape
        - **(D * layer수, batch_size, hidden_size)**
        - D: 양방향(bidirectional) 이면 2 아니면 1

        



**추론시 Input tensor 구조** <br>
- Input_data, (Hidden_state, Cell_state)
- Input_data의 shape
    - **(sequence_length, batch_size, input_feature_shape)**
- (Hidden_state, Cell_state)는 생략시 0 입력된다.
    - Hidden_state의 shape
        - **(D * layer수, batch_size, hidden_size)**
        - D: 양방향(bidirectional) 이면 2 아니면 1
    - Cell_state의 shape
        - **(D * layer수, batch_size, hidden_size)**
        - D: 양방향(bidirectional) 이면 2 아니면 1
### Output tentor 구조
- Output_data, (Hidden_state, Cell_state)
  
#### Output_data의 shape
- 모든 timestep의 출력결과를 묶어서 반환
- **(sequence length, batch_size, D * hidden_size)**
#### Hidden_state의 shape
- 마지막 timestep의 출력결과
- **(D * layer수, batch_size, hidden_size)**
- D: 양방향(bidirectional) 이면 2 아니면 1
#### Cell_state의 shape
- cell state(장기기억) 값
- **(D * layer수, batch_size, hidden_size)**
- D: 양방향(bidirectional) 이면 2 아니면 1

##### <b><span style='color:#F1A424'>01. 라이브러리 임포트 & 더미데이터 생성</span></b>

- `입력 : ( input, (hidden,cell)) ` <br> ➔ 첫번째 time step 에 입력할 hidden state, cell state = 생략 : 0
- `출력 : out1, (hidden1, cell1)` <br> 
    - `out1` : 타입step별 hidden state들
    - `(hidden1, cell1)` : 마지막 timestep 의 hidden & cell state


In [1]:
import torch
import torch.nn as nn

input_data = torch.randn(30,100,4)
# [30 : seq_len, 100 : batch_size, 4: feature 수]

In [ ]:
lstm1 = nn.LSTM(
    input_size=4,
    hidden_size =256 # 추출할 feature 수
)
out1, (hidden1, cell1) = lstm1(input_data)

print(out1.shape)
print(hidden1.shape)
print(cell1.shape)

In [ ]:
# 2. 양방향 lstm

lstm2 = nn.LSTM(
    input_size = 4,
    hidden_size = 256,
    bidirectional = True # 양방향
)

out2, (hidden2, cell2) = lstm2(input_data)

print(out2.shape) 
print(hidden2.shape)
print(cell2.shape)

torch.Size([30, 100, 512])
torch.Size([2, 100, 256])
torch.Size([2, 100, 256])


**따로따로 제공** <br>
- 1 레이어의 hiddenstate , cellstate
- 2 레이어의 hiddenstate, cellstate 
<br>
- hidden2의 0번 인덱스  정방향 
- hidden 2의 1번 인덱스  양방향

In [11]:
# 레이어를 쌓은 양방향 LSTM

lstm3 = nn.LSTM(
    input_size = 4,
    hidden_size = 256,
    num_layers = 3,
    bidirectional = True
)

out3, (hidden3, cell3) = lstm3(input_data)

print(out3.shape) 
print(hidden3.shape)
print(cell3.shape)

torch.Size([30, 100, 512])
torch.Size([6, 100, 256])
torch.Size([6, 100, 256])


---

## <b>2 <span style='color:#F1A424'>|</span> LSTM 감성분석 실습</b> 

- 긍정 부정 분류


`Pytorch의 nn.Embedding`
- Pytorch의 Embedding Layer는 word2vec과 마찬가지로 word embedding vector를 찾는 **Lookup Table**이다.
    - 단어의 **정수의 고유 index**가 입력으로 들어오면 Embedding Layer의 **그 index의 Vector**를 출력한다.
    - 모델이 학습되는 동안 모델이 풀려는 문제에 맞는 값으로 Embedding Layer의 vector들이 업데이트 된다.
    - Word2Vec의 embedding vector 학습을 nn.Embedding은 자신이 포함된 모델을 학습 하는 과정에서 한다고 생각하면 된다.

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.1 | </span></span></b>Pytorch의 nn.Embedding 예제</b></p></div>


In [2]:
import torch
import torch.nn as nn

embedding_layer = nn.Embedding(
    num_embeddings = 20000,
    embedding_dim= 100, # 임베딩 벡터의 차원수 ➔ 하나의 토큰을 몇개의
    padding_idx=0 # 패딩 토큰의 어휘사전에서의 idx
)

# 20_000 X 100 짜리 Embedding vector 생성 

In [3]:
w = embedding_layer.weight 
print(w.shape)

torch.Size([20000, 100])


In [ ]:
# 3번 토큰 단어의 임베딩 벡터를 조회
# ex. 3번 단어가 시계일 경우 시계에 대한 벡터임.
# 이걸 문맥에 맞추어 벡터를 바꾸어주는 게 lstm
w[2]

tensor([ 0.2836,  1.2627,  1.0634, -1.2989, -0.6355,  0.9968,  1.5015,  1.0175,
         0.9051, -1.0589,  0.2977,  0.0072, -0.9234, -0.2721,  0.3552,  0.1287,
         1.2777,  0.1016, -1.4408,  0.1369, -0.4074, -0.6271,  0.5415, -0.3278,
         0.8956,  1.3252, -0.6970, -0.4863,  1.1746,  0.6213, -0.4466, -1.3994,
        -1.8481, -1.3717, -0.4488, -0.5317,  0.7079,  0.3024,  0.1034,  1.7798,
        -1.0780, -1.9572,  0.1284, -0.2056,  0.3112,  0.0578, -0.1857,  0.4932,
         1.0034,  0.3707,  0.4040, -0.4982,  1.7899, -1.4416, -0.6907,  0.5845,
         0.1053,  0.5685,  0.4868,  0.2026,  1.1565,  0.9060,  0.6463, -0.2685,
        -0.5458,  0.2662, -0.3520,  0.4802,  1.1810,  0.2311,  0.6666,  0.0742,
        -0.5672, -0.4746,  0.8506, -0.0765,  0.1796,  0.2952,  0.0802,  0.2627,
         0.4665,  1.6957, -0.3945,  0.2125,  0.7260, -0.3740, -1.0379, -1.4326,
         0.1645,  1.0300, -0.7958,  0.3432,  0.5689,  1.9034,  0.4683, -0.6805,
         0.3249,  0.5259, -0.8490,  0.04

In [ ]:
# 0번 토큰 = padding 
w[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], grad_fn=<SelectBackward0>)

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.2 | </span></span></b>Korpora 데이터셋 로드 </b></p></div>

KORPORA 에서 nscm : naver 영화 댓글 데이터셋 가져오기 



In [11]:
from Korpora import Korpora
corpus = Korpora.load("nsmc")


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\bongr\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\bon

In [ ]:
all_inputs = corpus.get_all_texts()
all_labels = corpus.get_all_labels()

실제 댓글 원문:
('아 더빙.. 진짜 짜증나네요 목소리',
 '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다')
--------------------------------------------------
긍부정 라벨링 : # 1: 긍정 , 0 : 부정 
[0, 1, 0, 0, 1]


In [ ]:
print(all_inputs[:5])
print(all_labels[:5])
print("긍부정 라벨링 : # 1: 긍정 , 0 : 부정 ")
print(len(all_inputs))
print(type(corpus.train))

('아 더빙.. 진짜 짜증나네요 목소리', '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나', '너무재밓었다그래서보는것을추천한다', '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정', '사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다')
[0, 1, 0, 0, 1]
200000
<class 'Korpora.korpora.LabeledSentenceKorpusData'>


In [ ]:
corpus.train.texts[:5]
corpus.train.labels[:5]
corpus.train

[0, 1, 0, 0, 1]

In [20]:
corpus.test

NSMC.test: size=50000
  - NSMC.test.texts : list[str]
  - NSMC.test.labels : list[int]

**해석** : <br>
- NSMC.train.texts : list[str] ➔ X_trian
- NSMC.train.labels : list[int] ➔ y_train
---
- NSMC.test.texts : list[str] ➔ X_test 
- NSMC.test.labels : list[int] ➔ y_test

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.3 | </span></span></b>Kiwi py를 활용한 토큰화 </b></p></div>

**전처리 및 토큰화 단계** <br>
1. 영문 -> 소문자로 변환
2. 구두점 제거
3. 형태소 기반 토큰화
4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.

In [21]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [23]:
import string
import re

from kiwipiepy import Kiwi

kiwi = Kiwi()

def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text)
    text = [token.lemma for token in kiwi.tokenize(text)]
    
    return ' '.join(text)

In [14]:
print(Kiwi)

<class 'kiwipiepy.Kiwi'>


In [15]:
print(all_inputs[100])
text_preprocessing(all_inputs[100])

신카이 마코토의 작화와,미유와 하나카나가 연기를 잘해줘서 더대박이였다.


'신카이 마코토 의 작화 와 미유 와 하나카나 가 연기 를 잘 하다 어 주다 어서 더 대박 이다 였 다'

In [35]:
train_texts = corpus.train.texts
train_inputs = [text_preprocessing(txt) for txt in train_texts]

test_texts = corpus.test.texts
test_inputs = [text_preprocessing(txt) for txt in test_texts]

train_labels = corpus.train.labels
test_labels = corpus.test.labels

In [ ]:
import os
os.makedirs('data/nsmc')

train_data = {"text": train_inputs, "label": train_labels}
test_data = {"text": test_inputs, "label": test_labels}

import pickle
with open("data/nsmc/preprocessing_train.pkl", "wb") as fo:
    pickle.dump(train_data, fo)

with open("data/nsmc/preprocessing_test.pkl", "wb") as fo:
    pickle.dump(test_data, fo)

NameError: name 'train_data' is not defined

In [19]:
import pickle
with open("data/nsmc/preprocessing_train.pkl", "rb") as fi:
    train = pickle.load(fi)

with open("data/nsmc/preprocessing_test.pkl", "rb") as fi:
    test = pickle.load(fi)

train_inputs = train['text']
train_labels = train['label']

test_inputs = test['text']
test_labels = test['label']

In [36]:
all_inputs = train_inputs + test_inputs

In [27]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size = 30_000

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=5,
    special_tokens=["<pad>","<unk>"],
    continuing_subword_prefix="##"
)

tokenizer.train_from_iterator(all_inputs, trainer=trainer)


KeyboardInterrupt: 

In [28]:
tokenizer.get_vocab_size()

24856

In [31]:
# 저장
tokenizer.save("save_model/nsmc_bpe_tokenizer.json")

In [37]:
idx = 120
print(all_inputs[idx])
encoding = tokenizer.encode(all_inputs[idx])
print("토큰 ID")
print(encoding.ids)
print("토큰 문자열")
print(encoding.tokens)

중국인 특유 의 과장 허풍 있다 어 보이다 려고 안간힘 쓰다 ᆫ 노력 은 가상 하 나 고증 과 현실감 떨어지다 는 설정 이 거북 스럽 다 도대체 그 들 은 왜 이렇다 게 까지 스스로 를 과대 포장 하 는 것 이다 ᆫ지
토큰 ID
[9204, 6157, 2149, 6684, 17565, 5257, 1978, 5342, 5540, 1937, 3027, 4281, 5374, 59, 6094, 2135, 9046, 2815, 810, 8023, 622, 7397, 5575, 920, 5653, 2152, 7238, 5366, 946, 5643, 670, 1059, 2135, 2065, 5349, 580, 5313, 6771, 1284, 7760, 6372, 2815, 920, 574, 5251, 5332]
토큰 문자열
['중국인', '특유', '의', '과장', '허풍', '있다', '어', '보이다', '려고', '안', '##간', '##힘', '쓰다', 'ᆫ', '노력', '은', '가상', '하', '나', '고증', '과', '현실감', '떨어지다', '는', '설정', '이', '거북', '스럽', '다', '도대체', '그', '들', '은', '왜', '이렇다', '게', '까지', '스스로', '를', '과대', '포장', '하', '는', '것', '이다', 'ᆫ지']


In [38]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("save_model/nsmc_bpe_tokenizer.json")

In [39]:
# 인코딩 테스트
idx = 100
print(all_inputs[idx])

encode = tokenizer.encode(all_inputs[idx])

print(encode.tokens)
print(encode.ids)

신카이 마코토 의 작화 와 미유 와 하나카나 가 연기 를 잘 하다 어 주다 어서 더 대박 이다 였 다
['신카이', '마코토', '의', '작화', '와', '미', '##유', '와', '하나', '##카', '##나', '가', '연기', '를', '잘', '하다', '어', '주다', '어서', '더', '대박', '이다', '였', '다']
[20234, 18753, 2149, 9483, 2054, 1426, 3175, 2054, 5347, 3160, 3030, 529, 5277, 1284, 2177, 5254, 1978, 5267, 5260, 981, 5603, 5251, 2024, 946]


In [40]:
tokenizer.decode(encode.ids)

'신카이 마코토 의 작화 와 미 ##유 와 하나 ##카 ##나 가 연기 를 잘 하다 어 주다 어서 더 대박 이다 였 다'

In [41]:
print(encode.ids)

[20234, 18753, 2149, 9483, 2054, 1426, 3175, 2054, 5347, 3160, 3030, 529, 5277, 1284, 2177, 5254, 1978, 5267, 5260, 981, 5603, 5251, 2024, 946]


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.4 | </span></span></b>Dataset, Dataloader </b></p></div>

1. def __init__ : 

In [36]:
train_inputs[5]

'막 걸음마 떼다 ᆫ 3 세 부터 초등학교 1 학년 생 이다 ᆫ 8 살 용 영화 ㅋㅋㅋ 별 반 개 도 아깝다 음'

In [42]:
tokenizer.token_to_id("<pad>")

0

In [43]:
# 사용자 정의 Dataset 생성

import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - 댓글 리스트. 리스트에 댓글들을 담아서 받는다. ["댓글", "댓글", ...]
        labels: list - Label 리스트. (댓글의 긍부정 여부 - 긍정: 1, 부정: 0)
        max_length: 개별 댓글의 token 최대 개수. 모든 댓글의 토큰수를 max_length에 맞춘다.
        tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.labels = labels
        self.texts = [
            self.__pad_token_sequences(tokenizer.encode(txt).ids) for txt in texts
        ]    

    ###########################################################################################
    # id로 구성된 개별 문장 token list를 받아서 패딩 추가 [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    ############################################################################################
    def __pad_token_sequences(self, token_sequences):
        """
        token id로 구성된 개별 문서(댓글)의 token_id list를 받아서 max_length 길이에 맞추는 메소드
        max_length 보다 토큰수가 적으면 <pad> 토큰 추가, 많으면 max_length 크기로 줄인다.
            ex) max_length=5 이고 pad토큰 id가 0이라면
                [20, 2, 1] => [20, 2, 1, 0, 0]
                [20, 21, 30, 34, 60, 17, 21, 33] -> [20, 21, 30, 34, 60]
        """
    
        pad_token_id = self.tokenizer.token_to_id("<pad>")
    
        seq_len = len(token_sequences)
    
        if self.max_length < seq_len:
            result = token_sequences[:self.max_length]
        else:
            result = token_sequences + ([pad_token_id] * (self.max_length - seq_len))

        return result
        
    def __len__(self):
        
        return len(self.texts)


    def __getitem__(self, idx):
        """
        idx 번째 text와 label을 학습 가능한 type으로 변환해서 반환
        Parameter
            idx: int 조회할 index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) - 댓글 토큰_id 리스트, 정답 Label
        """
        comment = torch.tensor(self.texts[idx], dtype=torch.int64)
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
    
        return comment, label

In [45]:
max_length = 30
trainset = NSMCDataset(train_inputs, train_labels, max_length, tokenizer)
testset = NSMCDataset(test_inputs, test_labels, max_length, tokenizer)

len(trainset), len(testset)

(150000, 50000)

In [46]:
trainset[110]


(tensor([6300, 5260, 5367, 5271, 5252, 5253, 5548, 2000, 5287, 1993, 5298, 6093,
         2152, 5821, 5251,  839,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0]),
 tensor([0.]))

In [47]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=64)

In [48]:
len(train_loader), len(test_loader)

(2343, 782)

# 모델링

In [49]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Torch: 2.12.0+cu126
CUDA: 12.6
GPU 사용 가능: True
NVIDIA GeForce RTX 3060 Laptop GPU


In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [51]:
import torch
import torch.nn as nn

class NSMCClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1, bidirectional=True, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout
        )

        self.classifier = nn.Linear(
            in_features= hidden_size * 2 if bidirectional else hidden_size,
            out_features=1
        )

        self.sigmoid = nn.Sigmoid()


    def forward(self, X):
        """
        Args: 
            X(torch.Tensor): 입력 문서의 토큰 리스트. shape: [batch_size, seq_length(max_length)], [64, 30]
            연산 순서 -> embedding_model =>(transpose)=>lstm => classifier => sigmoid
        """
        embedding_vector = self.embedding(X)
        
        embedding_vector = embedding_vector.transpose(1, 0)

        out, _ = self.lstm(embedding_vector)

        output = self.classifier(out[-1])
        last_output = self.sigmoid(output)
        
        return last_output


# 모델 생성

In [52]:
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 100
hidden_size = 64
num_layers = 2
bidirectional = True
dropout = 0.3

In [53]:
# torch info로 모델 확인
from torchinfo import summary
dummy_data = torch.randint(1, 10, (64, max_length))
summary(
    NSMCClassifier(vocab_size, embedding_dim, hidden_size, num_layers, bidirectional, dropout),
    input_data=dummy_data
)

Layer (type:depth-idx)                   Output Shape              Param #
NSMCClassifier                           [64, 1]                   --
├─Embedding: 1-1                         [64, 30, 100]             2,485,600
├─LSTM: 1-2                              [30, 64, 128]             184,320
├─Linear: 1-3                            [64, 1]                   129
├─Sigmoid: 1-4                           [64, 1]                   --
Total params: 2,670,049
Trainable params: 2,670,049
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 512.98
Input size (MB): 0.02
Forward/backward pass size (MB): 3.50
Params size (MB): 10.68
Estimated Total Size (MB): 14.20

# 학습 trian/test 함수 정의 

In [54]:
# 1 에폭 학습 함수.
def train(model, dataloader, loss_fn, optimizer, device="cpu"):
    # 1. 모델을 train 모드로 변경
    model.train()
    # 2. 모델을 device로 이동
    model = model.to(device)

    # 1 에폭 학습
    train_loss = 0.0
    for X, y in dataloader:
        # 1 step 학습
        # 1. X, y를 device 이동
        X, y = X.to(device), y.to(device)

        # 2. 추론
        pred = model(X)

        # 3. loss 계산
        loss = loss_fn(pred, y)

        # 4. gradient 값 계산 - 오차역전파
        loss.backward()

        # 5. weight/bias(파라미터) 업데이트 = new_weight = weight.data - weight.grad * 학습율
        optimizer.step()

        # 6. grad 초기화
        optimizer.zero_grad()

        # loss값 누적
        train_loss += loss.item()

    return train_loss / len(dataloader) # 1 에폭 학습 loss를 반환.

In [55]:
@torch.no_grad
def eval(model, dataloader, loss_fn, device="cpu"):
    """모델 평가/검증 함수"""
    # 모델을 eval 모드로 변경. (평가/추론)
    model.eval()
    model.to(device)

    eval_loss, eval_acc = 0.0, 0.0
    
    for X,  y in dataloader:
        # 1. X, y를 device이동
        X, y = X.to(device), y.to(device)

        # 2. 추론  - 양성일 확률이 출력.
        pred_proba = model(X)
        pred_label = (pred_proba > 0.5).type(torch.int32)

        # 3. 평가(loss, accuracy)
        eval_loss += loss_fn(pred_proba, y).item()
        eval_acc += (pred_label == y).sum().item()  # 현재 step에서 몇개 맞았는지 대입.

    return eval_loss / len(dataloader), eval_acc / len(dataloader.dataset)

# train

In [56]:
lr = 0.0001
epochs = 3

model = NSMCClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    dropout=dropout
).to(device)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [57]:
from time import time
s = time()

train_loss_list = []
eval_loss_list = []
eval_acc_list = []

for epoch in range(epochs):
    train_loss = train(model, train_loader, loss_fn, optimizer, device)
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, device)
    train_loss_list.append(train_loss)
    eval_loss_list.append(eval_loss)
    eval_acc_list.append(eval_acc)
    print(train_loss, eval_loss, eval_acc, sep=" || ")

e = time()
print("학습에 걸린 시간:", (e-s), "초")

0.5609193605665361 || 0.4467024875190252 || 0.78824
0.4134440283349473 || 0.40475168044838455 || 0.81556
0.37730574873055445 || 0.38281490011593267 || 0.82744
학습에 걸린 시간: 112.36038374900818 초


# 모델 저장

In [59]:
torch.save(model, "save_model/nsmc_lstm_model.pt")

In [61]:
load_model = torch.load("save_model/nsmc_lstm_model.pt", weights_only=False)


# 학습시킨 모델가지고 추론해보깅

In [63]:
#전처리 함수 
# from konlpy.tag import Okt
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text)
    text = [token.lemma for token in kiwi.tokenize(text)]
    return ' '.join(text)

In [62]:
# 패딩 처리 메소드
def pad_token_sequences(token_sequences, max_length):
    """padding 처리 메소드."""
    pad_token = tokenizer.token_to_id('<pad>')  
    seq_length = len(token_sequences)           
    result = None
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [ ]:
def predict_data_preprocessing(text_list:list[str], max_length:int=30):
    """
    모델에 입력할 수 있는 input data를 생성
    Parameter:
        text_list: list - 추론할 댓글리스트 
    Return
        torch.LongTensor - 댓글 token_id tensor 
    """
    # 기본 전처리 ["댓글1","댓글2","댓글3"...] -> ["전처리된 댓글1", "전처리된 댓글2"]
    text_list = [text_preprocessing(txt) for txt in text_list]
    
    # 토큰화
    #["전처리된 댓글1", "전처리된 댓글2"] -> [[100,200,170,1,..]]
    token_list = [tokenizer.encode(txt).ids for txt in text_list]
    # encode 라는 타입에서 ids 만 뽑아내기 
    
    # 토큰개수 max_length에 맞추기
    # [[100,200,170,1,..]] -> pad 채워주기
    token_list = [pad_token_sequences(token, max_length) for token in token_list]
    
    return torch.tensor(token_list, dtype=torch.int64)

In [ ]:
comment_list = ["아 진짜 재미없다.", "여기 식당 먹을만 해요", "이걸 영화라고 만들었냐?", "기대 안하고 봐서 그런지 괜찮은데.", "이걸 영화라고 만들었나?", "아! 뭐야 진짜.", "재미있는데.", "연기 짱 좋아. 한번 더 볼 의향도 있다.", "뭐 그럭저럭"]
input_tensor = predict_data_preprocessing(comment_list, tokenizer, 30)
input_tensor.shape

torch.Size([9, 30])

In [72]:
input_tensor

tensor([[ 1935,  5278,  5336,   946,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 5707, 10648,  5496,  2137,  1310,  2815,  5268,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 5292,    63,  5252,  5251,  5329,  5283,  1993,   839,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 5354,  1937,  5254,   608,  5253,  5260,  5346,  5332,  5407,  5433,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [ 5292,    63,  5252,  5251,  5329,  528

**파이썬 : zip() 함수**
- 여러 개의 iterable한 객체를 인자로 받고 각 객체가 담고 있는 원소를 튜플 형태로 접근하는 반복문 반환 
- 여러 객체에 존재하는 같은 인덱스의 데이터를 하나씩 차례로 짝을 지어줌
- 주의사항 : zip() 인자의 길이가 다를 때는 가장 짧은 인자를 기준으로 엮이고 나머지는 버려짐!!

In [ ]:
proba = torch.tensor([0.1,0.7,0.9])
label = torch.where(proba > 0.5, 1, 0)
proba2 = [1-p if l==0 else p for l,p in zip(label,proba)]

for l, p in zip(label, proba):
    print(l,p)

tensor(0) tensor(0.1000)
tensor(1) tensor(0.7000)
tensor(1) tensor(0.9000)


In [87]:
# 추론 함수
@torch.no_grad

def predict(model, comment_list: torch.LongTensor):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    comment_list = comment_list.to(device)
    pred_proba = model(comment_list)
    pred_label = torch.where(pred_proba > 0.5,1,0) # 부정 : 0, # 긍정 : 1
    result_proba = [1-p if l ==0 else p for l,p in zip(pred_label, pred_proba)]
    
    return pred_label, result_proba
    

In [104]:
load_model = torch.load("save_model/nsmc_lstm_model.pt", weights_only= False)
p_label, p_proba = predict(load_model, input_tensor)

In [107]:
for txt, label, proba in zip(comment_list, p_label, p_proba):
    print(txt)
    
    print("긍정적" if label.item()==1 else "부정적")
    
    print(f"확률 : {proba.item():.2f}")
    print("-" * 50)
    

아 진짜 재미없다.
부정적
확률 : 0.98
--------------------------------------------------
여기 식당 먹을만 해요
부정적
확률 : 0.57
--------------------------------------------------
이걸 영화라고 만들었냐?
부정적
확률 : 0.98
--------------------------------------------------
기대 안하고 봐서 그런지 괜찮은데.
긍정적
확률 : 0.78
--------------------------------------------------
이걸 영화라고 만들었나?
부정적
확률 : 0.98
--------------------------------------------------
아! 뭐야 진짜.
부정적
확률 : 0.77
--------------------------------------------------
재미있는데.
긍정적
확률 : 0.84
--------------------------------------------------
연기 짱 좋아. 한번 더 볼 의향도 있다.
긍정적
확률 : 0.94
--------------------------------------------------
뭐 그럭저럭
부정적
확률 : 0.90
--------------------------------------------------


In [ ]:
with torch.no_grad():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_tensor = input_tensor.to(device)
    pred_proba = model(input_tensor)
    pred_label = (pred_proba > 0.5).type(torch.int32)
    for txt, pred_label in zip(comment_list, pred_label):
        print(txt, end=" = ")
        print(f"긍정적 댓글({pred_proba})" if pred_label.item()==1 else f"부정적 댓글({pred_proba})")

아 진짜 재미없다. = 부정적 댓글(tensor([[0.0226],
        [0.4333],
        [0.0183],
        [0.7762],
        [0.0215],
        [0.2286],
        [0.8401],
        [0.9392],
        [0.0988]], device='cuda:0'))
여기 식당 먹을만 해요 = 부정적 댓글(tensor([[0.0226],
        [0.4333],
        [0.0183],
        [0.7762],
        [0.0215],
        [0.2286],
        [0.8401],
        [0.9392],
        [0.0988]], device='cuda:0'))
이걸 영화라고 만들었냐? = 부정적 댓글(tensor([[0.0226],
        [0.4333],
        [0.0183],
        [0.7762],
        [0.0215],
        [0.2286],
        [0.8401],
        [0.9392],
        [0.0988]], device='cuda:0'))
기대 안하고 봐서 그런지 괜찮은데. = 긍정적 댓글(tensor([[0.0226],
        [0.4333],
        [0.0183],
        [0.7762],
        [0.0215],
        [0.2286],
        [0.8401],
        [0.9392],
        [0.0988]], device='cuda:0'))
이걸 영화라고 만들었나? = 부정적 댓글(tensor([[0.0226],
        [0.4333],
        [0.0183],
        [0.7762],
        [0.0215],
        [0.2286],
        [0.8401],
        [0.9392],
        [0.0988

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.1 | </span></span></b> GRU 와 LSTM과 차이</b></p></div>

- LSTM의 한계점 :
    1. 복잡하고 무거운 연산량 <br> 한 시점(time-step)마다 3개의 게이트를 제어, 셀 상태와 은닉상태(hidden state)를 모두 업데이트 즉 <br>
    4개의 레이어 (3개의 게이트 + 1개 셀상태 후보)
    2. 계산량이 너무 많다 = 파라미터가 많다. 

- LSTM (LSTM 경량화 버전)
    1. Reset 게이트 : 과거의 정보를 얼마나 지울지(비율) 
        1. 이전 시점 기억($h_{t-1}$) X $U_r$ + 현재 입력($x_t$) X $W_r$ <br> ➔ 시그모이드 함수 통과 ➔ 결과 : $r_t$ = [0-1]
        2.$h_{t-1}$ X $r_t$(과거의 기억 중 이번 예측 tanh에 얼마나 반영할지 비율 ➔ 0에 가까우면 과거를 더 많이 리셋)
    2. 후보 은닉 상태 ($h'_t$) : 현재 시점의 새로운 기억 후보
        1. 하단의 tanh(현재 입력($x_t$), 리셋 게이트가 적용된 과거 기억($r_t * h_{t-1}$)) ➔ 현재 시점의 새로운 기억 후보 

    3.  Update Gate ($z_t$)와 최종 출력 ($h_t$) : lstm의 forget과 input 게이트 역할을 동시에 수행
        1. update gate : 새로운 기억($h'_t$)을 얼마나 섞을지➔ 비율
        2. $z_t$ = 이전 시점 기억($h_{t-1}$) X $U_r$ + 현재 입력($x_t$) X $W_r$ ➔ 시그모이드에 넣어 나온 결과 : $z_t$ = [0-1]
        3. 과거 기억을 얼마나 가져갈지 ➔ 과거 기억 ($h_{t-1}$) X $z_t$
        4. 더할량 ➔ 새기억후보($h'_t$) X 1 - z_t$
        5. 마지막으로 두 값을 더하기(+)하여 최종 은닉 상태인 $h_t$를 출력
    4. 특징 : 
        1.  메모리 구조의 단일화 : cell state 제거. <br> ➔ hidden state 하나로만 기억과 출력 모두 관리 (더하기 곱하기만 한다! )
        2. 적은 파라미터 수 , 적은 연산량, lstm과 비슷한 성능
        3. 정리 :  $z_t$라는 하나의 게이트를 가지고 과거 기억을 보존할 비율($z_t$)와 현재 새로운 정볼르 반영할 비율($1 - z_t$)을 정함
        **즉 Zt 1개의 가중치 행렬로 다 해결**
